# 85 Statistics: R-012 reconciliation (D-31 / DV-02)
Post-result correction; D-31 remains Proposed. Preserves R-012 and frozen outer folds.
CPU Colab; no raw data needed. May take several hours. ETA appears after five permutations.
Keep the runtime connected. Drive checkpoints survive disconnection, but Colab may stop computing.
All analysis logic is in bbbd.reconcile_eye. Raw eye p is not a completed across-modality Holm test.


In [ ]:
import os, sys, tempfile, subprocess
from pathlib import Path
BRANCH = "a/r012-reconciliation"
REPO = Path(tempfile.mkdtemp(prefix="bbbd_reconcile_"))
def command(*args):
    subprocess.run(args, check=True, cwd=REPO)
command("git", "clone", "--branch", BRANCH, "--single-branch",
        "https://github.com/pbhatt0022/bbbd_mml.git", str(REPO))
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]")
command(sys.executable, "-m", "pytest", "-q", "tests/test_reconcile_eye.py")
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Code commit:", COMMIT)
print("Checkout:", REPO)


## Persistent backup
Mount Drive. Only a new timestamped folder under MyDrive/MML_BBBD/04_results/reconciliation is written.


In [ ]:
from google.colab import drive
from datetime import datetime, timezone
drive.mount("/content/drive")
BACKUP = Path("/content/drive/MyDrive/MML_BBBD/04_results/reconciliation") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BACKUP.mkdir(parents=True, exist_ok=False)
(BACKUP / "CODE_COMMIT.txt").write_text(COMMIT)
print("Backup:", BACKUP)


## Launch or resume
First run: RESUME = None. If interrupted in the same runtime, set RESUME to the printed
run directory and rerun only this cell. Do not rerun setup or change analysis settings.
A resume requires identical code, configuration, inputs and numerical package versions.
If the runtime is lost, retain the Drive backup and request restoration instructions.
No GitHub push or inference-log conclusion is automatic.


In [ ]:
import shutil, time
RESUME = None  # e.g. "results/R-013_eye_reconciliation" after interruption
for name in ("config.yaml", "docs/decision_log.md", "docs/deviations.md"):
    shutil.copy2(REPO / name, BACKUP / Path(name).name)
args = [sys.executable, "-u", "-m", "bbbd.reconcile_eye"]
if RESUME:
    args += ["--resume", str((REPO / RESUME).resolve())]
env = dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
env["PYTHONPATH"] = str(REPO / "src")
log = BACKUP / "console.log"
with log.open("a") as stream:
    process = subprocess.Popen(args, cwd=REPO, env=env, stdout=stream, stderr=subprocess.STDOUT)
    offset = 0
    while True:
        time.sleep(15)
        with log.open() as reader:
            reader.seek(offset)
            print(reader.read(), end="", flush=True)
            offset = reader.tell()
        for folder in (REPO / "results").glob("R-*_eye_reconciliation"):
            shutil.copytree(folder, BACKUP / folder.name, dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns("*.tmp.json"))
        shutil.copy2(REPO / "docs/run_registry.csv", BACKUP / "run_registry.csv")
        if process.poll() is not None:
            break
# Final mirror after the process exits, so the bundle includes final artifacts.
for folder in (REPO / "results").glob("R-*_eye_reconciliation"):
    shutil.copytree(folder, BACKUP / folder.name, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("*.tmp.json"))
shutil.copy2(REPO / "docs/run_registry.csv", BACKUP / "run_registry.csv")
if process.returncode:
    raise RuntimeError(f"Run stopped; checkpoint/log saved in {BACKUP}. Do not change thresholds.")
archive = shutil.make_archive(str(BACKUP.parent / BACKUP.name), "zip", BACKUP)
print("\nCompleted. Bundle:", archive)
from google.colab import files
files.download(archive)


## After completion
Keep the ZIP and Drive copy. Share the run ID and final output for interpretation and review.
Always pair p with its CI; across-modality Holm remains pending. Preserve R-012.
Save this notebook with outputs for submission. An in-progress registry entry is not a
completed result: eye_reconciliation.json must have status complete.
